# [KAGGLE] METC GRU training

Notebook dùng artifacts preprocessing đã hợp lệ. Bật GPU T4 x2, Internet và Files persistence trước khi chạy.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa được bật. Mở Notebook Settings và chọn GPU T4 x2.")
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
expected_arch = f"sm_{capability[0]}{capability[1]}"
supported_arches = torch.cuda.get_arch_list()
print("PyTorch:", torch.__version__)
print("GPU:", gpu_name, expected_arch)
if supported_arches and expected_arch not in supported_arches:
    raise RuntimeError(f"GPU {gpu_name} ({expected_arch}) không tương thích; hãy chọn T4 x2.")
probe = torch.ones(1, device="cuda") + 1
torch.cuda.synchronize()
print("CUDA preflight passed:", float(probe.item()))

In [ ]:
repo_dir = Path("/kaggle/working/DHB")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/reimei2009/DHB.git", str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", "main"], check=True)
commit = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", commit)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_training_model.py", "tests/test_training_data.py", "tests/test_cv_summary.py"], cwd=repo_dir, check=True)
print("GRU, fold và checkpoint smoke tests passed")

In [ ]:
feature_files = list(Path("/kaggle/input").rglob("features.parquet"))
if len(feature_files) != 1:
    raise RuntimeError(f"Cần đúng một features.parquet, hiện tìm thấy: {feature_files}")
data_dir = feature_files[0].parent
required = ["features.parquet", "window_index.parquet", "feature_manifest.json", "validation_report.json"]
missing = [name for name in required if not (data_dir / name).exists()]
if missing:
    raise FileNotFoundError(f"Kaggle Dataset thiếu file: {missing}")
report = json.loads((data_dir / "validation_report.json").read_text(encoding="utf-8"))
if report.get("valid") is not True:
    raise RuntimeError("validation_report.json không có valid=true")
print("Data directory:", data_dir)
print(json.dumps(report, ensure_ascii=False, indent=2))

## [KAGGLE] Chạy một thí nghiệm

Mặc định chạy ablation `shape+motion`. Sau khi có đủ bốn ablation, sửa `SELECTED_GROUPS` và lần lượt chọn `regularized`, `attention`.

In [ ]:
EXPERIMENT = "shape_motion"  # shape_motion | regularized | attention
SELECTED_GROUPS = ["shape", "relation", "motion"]
experiments = {
    "shape_motion": ("training.yaml", "dhb_gru_shape_motion", ["shape", "motion"]),
    "regularized": ("training_regularized.yaml", "dhb_gru_regularized", SELECTED_GROUPS),
    "attention": ("training_attention.yaml", "dhb_gru_attention", SELECTED_GROUPS),
}
config_name, output_name, groups = experiments[EXPERIMENT]
output_dir = Path("/kaggle/working") / output_name
RESUME_CHECKPOINT = None  # Đặt đường dẫn cụ thể nếu attach Previous Notebook Output.
resume_value = str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else "auto"
command = [
    sys.executable, str(repo_dir / "train.py"),
    "--config", str(repo_dir / "configs" / config_name),
    "--data-dir", str(data_dir),
    "--output-dir", str(output_dir),
    "--groups", *groups,
    "--resume", resume_value,
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=repo_dir, check=True)

In [ ]:
from IPython.display import Image, display
result = json.loads((output_dir / "result.json").read_text(encoding="utf-8"))
print(json.dumps(result, ensure_ascii=False, indent=2))
display(Image(filename=str(output_dir / "training_history.png")))
display(Image(filename=str(output_dir / "confusion_matrix.png")))

## [KAGGLE] 5-fold sau khi đã khóa cấu hình

Chỉ đổi `RUN_FOLDS=True` sau khi đã chọn `regularized` hoặc `attention`. Test gốc luôn bị loại khỏi năm lượt train.

In [ ]:
RUN_FOLDS = False
WINNER = "attention"  # regularized | attention
FOLD_GROUPS = SELECTED_GROUPS
fold_outputs = []
if RUN_FOLDS:
    fold_config = experiments[WINNER][0]
    for fold in range(5):
        fold_output = Path("/kaggle/working") / f"dhb_gru_{WINNER}_fold_{fold}"
        fold_outputs.append(fold_output)
        fold_command = [sys.executable, str(repo_dir / "train.py"), "--config", str(repo_dir / "configs" / fold_config), "--data-dir", str(data_dir), "--output-dir", str(fold_output), "--groups", *FOLD_GROUPS, "--fold", str(fold), "--resume", "auto"]
        print("Running fold:", " ".join(fold_command))
        subprocess.run(fold_command, cwd=repo_dir, check=True)

In [ ]:
if RUN_FOLDS:
    summary_dir = Path("/kaggle/working/dhb_gru_cv_summary")
    summary_command = [sys.executable, str(repo_dir / "summarize_cv.py"), *[str(path) for path in fold_outputs], "--output-dir", str(summary_dir)]
    subprocess.run(summary_command, cwd=repo_dir, check=True)

## [KAGGLE] Test cuối cùng

Chỉ bật sau khi feature và kiến trúc đã khóa. Cell dùng checkpoint split chính và không train thêm epoch.

In [ ]:
RUN_FINAL_TEST = False
FINAL_WINNER = "attention"  # regularized | attention
if RUN_FINAL_TEST:
    final_config, final_output_name, final_groups = experiments[FINAL_WINNER]
    final_output = Path("/kaggle/working") / final_output_name
    final_command = [sys.executable, str(repo_dir / "train.py"), "--config", str(repo_dir / "configs" / final_config), "--data-dir", str(data_dir), "--output-dir", str(final_output), "--groups", *final_groups, "--resume", "auto", "--evaluate-only", "--evaluate-test"]
    print("Final test:", " ".join(final_command))
    subprocess.run(final_command, cwd=repo_dir, check=True)